# Week 2b — Adding oxidation-state features
**Why:** the Week 2 error analysis showed two groups of failures that composition features cannot see:
1. **Anode-type hosts** such as graphite (C) and low-voltage oxides — predicted far too high
2. **Hosts whose metal is already at or beyond its highest common oxidation state** (MoO4, SbO3, CrF6 …) — predicted far too low, because their high DFT voltage comes from oxygen/fluorine redox

**What we add (10 features), all from the formulas by charge balance:**
- average oxidation state of the redox-active metals in the **charged** host and the **discharged** material, and the change between them (electrons per metal)
- the highest *common* oxidation state of those metals, and how far the charged host goes **beyond** it (`ox_excess`, `ox_above_max`)
- a flag for formulas whose oxidation state comes out unphysical (`ox_unreliable`; the value is then treated as missing)
- flags for hosts with **no redox metal**, **elemental hosts** (e.g. graphite) and **anion-free hosts**

**Fair comparison:** same data, same framework-based train/test split (same random seed) and same XGBoost settings as Week 2 — the only change is the new features.

In [ ]:
!pip -q install pymatgen xgboost
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/electrode_voltage_ml"

In [ ]:
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

ION_ORDER = ["Li", "Na", "K", "Mg", "Ca", "Zn"]
PALETTE = dict(zip(ION_ORDER, sns.color_palette("tab10", len(ION_ORDER))))

feat = pd.read_csv(f"{DATA_DIR}/features_week2.csv")                 # Week 2 features (same row order)
clean = pd.read_csv(f"{DATA_DIR}/electrodes_clean.csv")
clean = clean[clean["working_ion"].isin(ION_ORDER)].reset_index(drop=True)
assert (clean["framework_formula"].values == feat["framework"].values).all(), "row order mismatch"
week2 = json.load(open(f"{DATA_DIR}/xgb_params.json"))
print(len(feat), "electrodes | Week 2 best settings:", week2["best_params"])

## 1. Oxidation-state features (charge balance)

In [ ]:
import numpy as np
from pymatgen.core import Composition, Element

ANION_Q = {"O": -2, "F": -1, "S": -2, "Se": -2, "Te": -2, "N": -3, "Cl": -1, "Br": -1, "I": -1}
# Elements that form positively charged centres of oxo-anions when oxygen is present
OXO_Q = {"P": 5, "Si": 4, "B": 3, "As": 5, "C": 4, "S": 6, "Se": 6, "N": 5, "Cl": 7, "I": 5, "Te": 6, "Ge": 4}
FIXED_Q = {"H": 1, "Li": 1, "Na": 1, "K": 1, "Rb": 1, "Cs": 1, "Be": 2, "Mg": 2, "Ca": 2, "Sr": 2, "Ba": 2,
           "Zn": 2, "Cd": 2, "Al": 3, "Ga": 3, "Sc": 3, "Y": 3, "La": 3, "Ce": 3, "Pr": 3, "Nd": 3,
           "Sm": 3, "Eu": 3, "Gd": 3, "Tb": 3, "Dy": 3, "Ho": 3, "Er": 3, "Tm": 3, "Yb": 3, "Lu": 3}
P_REDOX = {"Sb", "Sn", "Bi", "Pb", "Tl", "In"}
OX_RANGE = (-4.0, 8.0)          # physically sensible range; values outside -> unreliable

def _common_max(sym):
    try:
        cs = Element(sym).common_oxidation_states
        return max(cs) if cs else np.nan
    except Exception:
        return np.nan

def redox_state(formula):
    """Average oxidation state of redox-active metals by charge balance; NaN if unreliable."""
    comp = Composition(formula)
    syms = {str(e) for e in comp.elements}
    has_O = "O" in syms
    charge, redox = 0.0, {}
    for e in comp.elements:
        s, n = str(e), comp[e]
        if has_O and s in OXO_Q and s != "O":
            charge += OXO_Q[s] * n
        elif s in ANION_Q:
            charge += ANION_Q[s] * n
        elif s in FIXED_Q:
            charge += FIXED_Q[s] * n
        elif e.is_transition_metal or s in P_REDOX:
            redox[s] = n
        # anything else (elemental C, Si, Ge, P without O ...) contributes 0
    n_redox = sum(redox.values())
    if n_redox == 0:
        return np.nan, np.nan, redox
    avg = -charge / n_redox
    cmax = sum(_common_max(s) * n for s, n in redox.items()) / n_redox
    if not (OX_RANGE[0] <= avg <= OX_RANGE[1]):
        return np.nan, cmax, redox          # unphysical -> let the model treat it as missing
    return avg, cmax, redox

def ox_features(framework, discharged):
    comp = Composition(framework)
    syms = [str(e) for e in comp.elements]
    anion_free = not any(s in ANION_Q for s in syms)
    ox_c, cmax, redox = redox_state(framework)
    ox_d = redox_state(discharged)[0] if isinstance(discharged, str) else np.nan
    change = ox_c - ox_d if not (np.isnan(ox_c) or np.isnan(ox_d)) else np.nan
    if not np.isnan(change) and not (0 < change <= 4):
        change = np.nan                      # more than 4 electrons per metal is not physical
    excess = ox_c - cmax if not np.isnan(ox_c) else np.nan
    return {
        "ox_charged": ox_c,
        "ox_discharged": ox_d,
        "ox_change": change,
        "ox_common_max": cmax,
        "ox_excess": excess,
        "ox_above_max": float(excess > 0.2) if not np.isnan(excess) else 0.0,
        "ox_unreliable": float(len(redox) > 0 and np.isnan(ox_c)),
        "no_redox_metal": float(len(redox) == 0),
        "elemental_host": float(len(syms) == 1),
        "anion_free_host": float(anion_free),
    }


In [ ]:
cache = {}
rows = []
for fw, dis in zip(clean["framework_formula"], clean["formula_discharge"]):
    key = (fw, dis)
    if key not in cache:
        try:
            cache[key] = ox_features(fw, dis)
        except Exception:
            cache[key] = {k: np.nan for k in ["ox_charged", "ox_discharged", "ox_change", "ox_common_max",
                                              "ox_excess", "ox_above_max", "ox_unreliable", "no_redox_metal",
                                              "elemental_host", "anion_free_host"]}
    rows.append(cache[key])
OX = pd.DataFrame(rows)
print("Hosts beyond common max oxidation state:", int(OX["ox_above_max"].sum()), "(normal for charged cathodes, e.g. Co4+, Ni4+)")
print("Unreliable (unphysical) oxidation states set to missing:", int(OX["ox_unreliable"].sum()))
print("Elemental hosts:", int(OX["elemental_host"].sum()), "| hosts without redox metal:", int(OX["no_redox_metal"].sum()))
OX.describe().T[["mean", "min", "max"]].round(2)

## 2. Same split, same model — Week 2 features vs. Week 2 + oxidation features

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

y = feat["average_voltage"].values
ions = feat["working_ion"].values
groups = feat["framework"].values
X_old = feat.drop(columns=["average_voltage", "working_ion", "framework"])
X_new = pd.concat([X_old, OX], axis=1)

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X_old, y, groups))
def scores(t, p):
    return {"MAE (V)": mean_absolute_error(t, p), "RMSE (V)": np.sqrt(mean_squared_error(t, p)), "R2": r2_score(t, p)}

def fit_predict(X):
    m = XGBRegressor(**week2["best_params"], min_child_weight=2, reg_lambda=1.0, random_state=0, n_jobs=-1)
    m.fit(X.iloc[tr], y[tr]); return m, m.predict(X.iloc[te])

m_old, p_old = fit_predict(X_old)
m_new, p_new = fit_predict(X_new)
comp = pd.DataFrame({"Week 2 (composition)": scores(y[te], p_old),
                     "Week 2b (+ oxidation states)": scores(y[te], p_new)}).T.round(3)
comp

In [ ]:
rows = []
for i in ION_ORDER:
    m = ions[te] == i
    if m.sum() > 5:
        rows.append({"ion": i, "n_test": int(m.sum()),
                     "MAE week 2": mean_absolute_error(y[te][m], p_old[m]),
                     "MAE week 2b": mean_absolute_error(y[te][m], p_new[m])})
per_ion = pd.DataFrame(rows).set_index("ion")
per_ion["change (V)"] = per_ion["MAE week 2b"] - per_ion["MAE week 2"]
per_ion.round(3)

## 3. Did it fix the Week 2 outliers?

In [ ]:
err = pd.DataFrame({"framework": groups[te], "ion": ions[te], "DFT_V": y[te],
                    "pred_week2": p_old, "pred_week2b": p_new})
err["err_week2"] = (err["DFT_V"] - err["pred_week2"]).abs()
err["err_week2b"] = (err["DFT_V"] - err["pred_week2b"]).abs()
print("The 10 worst cases from Week 2, before and after:")
err.sort_values("err_week2", ascending=False).head(10).round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5.2), sharex=True, sharey=True)
for a, p, title in [(ax[0], p_old, "Week 2"), (ax[1], p_new, "Week 2b (+ oxidation states)")]:
    for i in ION_ORDER:
        m = ions[te] == i
        a.scatter(y[te][m], p[m], s=12, alpha=0.7, color=PALETTE[i], label=i)
    a.plot([-0.2, 6.2], [-0.2, 6.2], "k--", lw=1)
    a.set(xlim=(-0.2, 6.2), ylim=(-0.2, 6.2), xlabel="DFT average voltage (V)",
          title=f"{title}  (MAE = {mean_absolute_error(y[te], p):.2f} V)")
ax[0].set_ylabel("Predicted voltage (V)"); ax[1].legend(title="Ion")
plt.tight_layout(); plt.savefig("fig_parity_week2_vs_2b.png", dpi=200); plt.show()

In [ ]:
imp = pd.Series(m_new.get_booster().get_score(importance_type="gain")).sort_values(ascending=False).head(20)
imp = imp / imp.sum()
colors = ["#C44E52" if k.startswith("ox_") or k in ("no_redox_metal", "elemental_host", "anion_free_host", "ox_unreliable") else "#4C72B0"
          for k in imp.index[::-1]]
imp[::-1].plot.barh(figsize=(7, 6), color=colors, title="Top 20 features, Week 2b (red = new oxidation features)")
plt.xlabel("Relative importance"); plt.tight_layout(); plt.savefig("fig_feature_importance_2b.png", dpi=200); plt.show()

## 4. Save

In [ ]:
import shutil
m_new.save_model("xgb_voltage_week2b.json")
pd.concat([feat, OX], axis=1).to_csv("features_week2b.csv", index=False)
err.to_csv("test_predictions_week2b.csv", index=False)
comp.to_csv("results_week2b.csv"); per_ion.to_csv("results_per_ion_week2b.csv")
for f in ["xgb_voltage_week2b.json", "features_week2b.csv", "test_predictions_week2b.csv", "results_week2b.csv",
          "results_per_ion_week2b.csv", "fig_parity_week2_vs_2b.png", "fig_feature_importance_2b.png"]:
    shutil.copy(f, DATA_DIR)
print("Saved to", DATA_DIR)

## What to send me
1. The **comparison table** (Week 2 vs Week 2b)
2. The **per-ion change** table
3. The **worst-cases before/after** table
4. The two plots

Even a modest drop in overall MAE is meaningful if the specific outliers (graphite, MoO4, SbO3 …) improve — that shows the new features capture real chemistry.